# Profiling Academic Trajectories: A ML Exploration of Student Performance

*Jairik "JJ" McCauley*

---

## Introduction

The Higher Education Students Performance Evaluation dataset [1] contains a mix of demographic, socioeconomic, academic, and behavioral attributes collected from students. Due to the wide array of features (32 in total), there is an immense amount of flexibility for analysis and exploration. Understanding what influences student performance has implications for advising, resource allocation, and instructional design within higher education.

In this project, I aim to identify different clusters of students, apply classification models to predict GPA categories, and gain insight into how individual features may impact perceived preparation and interest in a chosen field. These analyses enable exploration of not only predictive performance, but also broader patterns that may support student success and institutional planning.

[1]: Yilmaz, N. & Şekeroğlu, B. (2019). Higher Education Students Performance Evaluation [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C51G82


---

## Data Exploration

Within the following sections, we will be understanding the features of the dataset, describing it using scientific methods, wrangling it for clean analysis, and observing feature relationships.

### Understanding the Dataset

Prior to diving into this large dataset, it is important to understand the overall structure with each feature and type in the dataset:

| #  | Feature                              | Type                                  |
| -- | ------------------------------------ | ------------------------------------- |
| 1  | Student Age                          | Categorical (ordinal)                 |
| 2  | Sex                                  | Binary                                |
| 3  | Graduated High-School Type           | Categorical                           |
| 4  | Scholarship Type                     | Categorical (ordinal)                 |
| 5  | Additional Work                      | Binary                                |
| 6  | Artistic/Sports Activity             | Binary                                |
| 7  | Has a Partner                        | Binary                                |
| 8  | Total Salary                         | Categorical (ordinal)                 |
| 9  | Transportation Mode                  | Categorical                           |
| 10 | Accommodation Type                   | Categorical                           |
| 11 | Mother’s Education                   | Categorical (ordinal)                 |
| 12 | Father’s Education                   | Categorical (ordinal)                 |
| 13 | Number of Siblings                   | Categorical (ordinal)                 |
| 14 | Parental Status                      | Categorical                           |
| 15 | Mother’s Occupation                  | Categorical                           |
| 16 | Father’s Occupation                  | Categorical                           |
| 17 | Weekly Study Hours                   | Categorical (ordinal)                 |
| 18 | Reading Frequency (Non-Scientific)   | Categorical (ordinal)                 |
| 19 | Reading Frequency (Scientific)       | Categorical (ordinal)                 |
| 20 | Attends Seminars/Conferences         | Binary                                |
| 21 | Impact of Projects/Activities        | Categorical                           |
| 22 | Class Attendance                     | Categorical (ordinal)                 |
| 23 | Midterm Prep Method                  | Categorical                           |
| 24 | Midterm Prep Timing                  | Categorical                           |
| 25 | Takes Notes in Class                 | Categorical (ordinal)                 |
| 26 | Listens in Class                     | Categorical (ordinal)                 |
| 27 | Discussion Improves Interest/Success | Categorical (ordinal)                 |
| 28 | Flip Classroom Usefulness            | Categorical                           |
| 29 | Cumulative GPA (Last Semester)       | Categorical (ordinal)                 |
| 30 | Expected Cumulative GPA (Graduation) | Categorical (ordinal)                 |
| 31 | Course ID                            | Identifier / Categorical              |
| 32 | Output Grade (Letter Grade)          | Target Variable (Ordinal Categorical) |

From the dataset documentation, we can see that features 1-10 are personal questions, 11-16 are family questions, and 17-32 include educational habits. Additionally, it is important to note that this is a relatively small dataset, containing only around 145 instances. Therefore, generalization will be an important consideration throughout this analysis and exploration.


### Describing the Dataset

Firstly, we will load in the data using UCI's ML library.

In [3]:
uv pip install ucimlrepo  # Install the ucimlrepo package, if not already installed (not a standard library)

Using Python 3.12.3 environment at: /mnt/c/Users/JJ/Desktop/Repos/Data-Science-Fundementals/.venv
Audited 1 package in 557ms
Note: you may need to restart the kernel to use updated packages.


In [24]:
from ucimlrepo import fetch_ucirepo  # Fetching the dataset
import pandas as pd  # Data manipulation
import numpy as np  # Numerical operations

# Fetch this dataset from the UCI Machine Learning Repository
higher_education_students_performance = fetch_ucirepo(id=856)

# Extract the full data, metadata, and variable information
hesp_data_raw = higher_education_students_performance.data
hesp_metadata = higher_education_students_performance.metadata
hesp_variable_info = higher_education_students_performance.variable_info

# Convert the raw data into a pandas DataFrame for easier manipulation
hesp_df_raw = pd.concat([hesp_data_raw.features, hesp_data_raw.targets], axis=1)
type(hesp_df_raw)

<class 'pandas.core.frame.DataFrame'>

Now, we can use this dataframe to gain actionable insights on the dataframe. Firstly, we can view the metadata and variable info of the dataset, as provided by UCI.

In [5]:
from pprint import pprint  # For pretty-printing

# Display metadata
print("=== Dataset Metadata ===")
pprint(hesp_metadata)

print("\n=== Variable Information ===")
print(hesp_variable_info)

=== Dataset Metadata ===
{'abstract': 'The data was collected from the Faculty of Engineering and '
             'Faculty of Educational Sciences students in 2019. The purpose is '
             "to predict students' end-of-term performances using ML "
             'techniques.',
 'additional_info': {'citation': 'YÄ±lmaz N., Sekeroglu B. (2020) Student '
                                 'Performance Classification Using Artificial '
                                 'Intelligence Techniques. In: Aliev R., '
                                 'Kacprzyk J., Pedrycz W., Jamshidi M., '
                                 'Babanli M., Sadikoglu F. (eds) 10th '
                                 'International Conference on Theory and '
                                 'Application of Soft Computing, Computing '
                                 'with Words and Perceptions - ICSCCW-2019. '
                                 'ICSCCW 2019. Advances in Intelligent Systems '
                                

Additionally, we can confirm the shape and datatypes.

In [6]:
print(f"Shape: {hesp_df_raw.shape}")  # Display dataframe shape
pprint(hesp_df_raw.info())  # Display dataframe info

Shape: (145, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 32 columns):
 #   Column                                                             Non-Null Count  Dtype
---  ------                                                             --------------  -----
 0   Student Age                                                        145 non-null    int64
 1   Sex                                                                145 non-null    int64
 2   Graduated high-school type                                         145 non-null    int64
 3   Scholarship type                                                   145 non-null    int64
 4   Additional work                                                    145 non-null    int64
 5   Regular artistic or sports activity                                145 non-null    int64
 6   Do you have a partner                                              145 non-null    int64
 7   Total salary if available  

As we can see from the info description (and dataset documentation), almost all columns are categorical, but represented by an integer scale. This is helpful, as we can skip any label encoding steps. 

The last observation that should be made regards missing values. Although the documentation states that there are no missing values, we can easily confirm this.

In [7]:
print(f"Null Values: {hesp_df_raw.isnull().sum().sum()}")  # Check for all null/missing values and print total count
print(f"NaN Values: {hesp_df_raw.isna().sum().sum()}")  # Check for all nan values and print total count
print(f"Empty String Values: {(hesp_df_raw == '').sum().sum()}")  # Check for all empty string values and print total count

Null Values: 0
NaN Values: 0
Empty String Values: 0


As we can see from these checks (and as the documentation confirms), there are no observable missing values.

### Data Wrangling

This dataset has conveniently came with numerical representations of categorical values and with no numerical values.

Additionally, it is important to fully understand the original meaning of the labels for each column. This is clearly outlined in the dataset's documentation, and can be easily represented by the following table:

<!-- Large table describing the numerical meanings behind the labels for each column -->
| Column | Possible Labels | Label meanings |
|---|---|---|
| Student Age | 1, 2, 3 | Student’s age group: 1 = 18–21; 2 = 22–25; 3 = ≥26 years. |
| Sex | 1, 2 | Biological sex: 1 = female; 2 = male. |
| High School Type | 1, 2, 3 | Type of high school attended: 1 = private; 2 = public/state; 3 = other. |
| Scholarship Type | 1–5 | Financial aid level: 1 = none; 2 = 25%; 3 = 50%; 4 = 75%; 5 = full scholarship. |
| Additional Work | 1, 2 | Whether student works while studying: 1 = yes; 2 = no. |
| Artistic/Sports Activity | 1, 2 | Participation in extracurricular arts or athletics: 1 = yes; 2 = no. |
| Partnership Status | 1, 2 | Whether student currently has a partner: 1 = yes; 2 = no. |
| Monthly Salary (if working) | 1–5 | Monthly income bracket: 1 = $135–200; 2 = $201–270; 3 = $271–340; 4 = $341–410; 5 = >$410. |
| Transportation Mode | 1–4 | Means of commuting to school: 1 = bus; 2 = car/taxi; 3 = bicycle; 4 = other. |
| Accommodation Type | 1–4 | Living situation while studying: 1 = rental; 2 = dormitory; 3 = family home; 4 = other. |
| Mother’s Education | 1–6 | Highest qualification: 1 = primary; 2 = secondary; 3 = high school; 4 = university degree; 5 = MSc; 6 = PhD. |
| Father’s Education | 1–6 | Same encoding as mother’s education. |
| Number of Siblings | 1–5 | Number of siblings: 1 = one; 2 = two; 3 = three; 4 = four; 5 = five or more. |
| Parental Status | 1–3 | Family background: 1 = married parents; 2 = divorced; 3 = parental loss. |
| Mother’s Occupation | 1–6 | Employment category: 1 = retired; 2 = housewife; 3 = government employee; 4 = private sector; 5 = self-employed; 6 = other. |
| Father’s Occupation | 1–5 | Employment category: 1 = retired; 2 = government employee; 3 = private sector; 4 = self-employed; 5 = other. |
| Weekly Study Hours | 1–5 | Hours spent studying per week: 1 = none; 2 = < 5 hrs; 3 = 6–10 hrs; 4 = 11–20 hrs; 5 = > 20 hrs. |
| Recreational Reading | 1–3 | Frequency of reading non-academic texts: 1 = never; 2 = sometimes; 3 = often. |
| Academic Reading | 1–3 | Frequency of reading scientific/academic texts: 1 = never; 2 = sometimes; 3 = often. |
| Seminar/Conference Attendance | 1, 2 | Participation in academic seminars: 1 = yes; 2 = no. |
| Impact of Projects/Activities | 1–3 | Perceived effect of participation on academic success: 1 = positive; 2 = negative; 3 = neutral. |
| Course Attendance | 1–3 | How consistently student attends classes: 1 = always; 2 = sometimes; 3 = never. |
| Exam Preparation Style | 1–3 | Preparation approach for first midterm: 1 = individually; 2 = with peers; 3 = not applicable. |
| Exam Preparation Timing | 1–3 | Timing of preparation: 1 = only right before exams; 2 = regularly; 3 = never. |
| Note-Taking Frequency | 1–3 | How often student takes notes: 1 = never; 2 = sometimes; 3 = always. |
| Listening in Classes | 1–3 | In-class attentiveness: 1 = never; 2 = sometimes; 3 = always. |
| Effect of Discussion | 1–3 | Whether discussion enhances learning: 1 = never; 2 = sometimes; 3 = always. |
| Perceived Usefulness of Flipped Classroom | 1–3 | View of flipped-learning method: 1 = not useful; 2 = useful; 3 = not applicable. |
| Previous semester GPA (/4) | 1–5 | GPA category: 1 = <2.00; 2 = 2.00–2.49; 3 = 2.50–2.99; 4 = 3.00–3.49; 5 = >3.49. |
| Expected Graduation GPA (/4) | 1–5 | Expected GPA category, same encoding as above. |
| Course ID | text/integer | Identifier representing the specific course taken by student. |
| Final Course Grade (Target) | 0–7 | Letter-grade category: 0 = Fail; 1 = DD; 2 = DC; 3 = CC; 4 = CB; 5 = BB; 6 = BA; 7 = AA. |


Through these observations, it becomes apparent that the scales are **not of equal magnitude**. This would lead to some features being weighted more heavily. For example, weekly study hours (scaled 1-5) would be scaled more heavily than seminar/conference attendance (scaled 1-2). Therefore, we should standardize all features before continuing.

In [30]:
from sklearn.preprocessing import StandardScaler  # For feature scaling

scaler = StandardScaler()  # Initialize the scaler

# Scale all columns with numerical data types
hesp_df_scaled = pd.DataFrame(
    scaler.fit_transform(hesp_df_raw),
    columns=hesp_df_raw.columns
)

hesp_df_scaled.head()  # Display the first few rows of the scaled dataframe

,Student Age,Sex,Graduated high-school type,Scholarship type,Additional work,Regular artistic or sports activity,Do you have a partner,Total salary if available,Transportation to the university,Accomodation type in Cyprus,...,Preparation to midterm exams 1,Preparation to midterm exams 2,Taking notes in classes,Listening in classes,Discussion improves my interest and success in the course,Flip-classroom,Cumulative grade point average in the last semester (/4.00),Expected Cumulative grade point average in the graduation (/4.00),Course ID,OUTPUT Grade
0,0.620766,0.816497,1.970956,-0.712873,-1.399708,0.816497,0.852168,-0.617265,-0.586970,-0.935675,...,-0.551503,-0.406604,0.808493,-0.082052,-2.313146,0.239081,-1.638252,-1.887666,-0.963726,-1.017122
1,0.620766,0.816497,1.970956,-0.712873,-1.399708,0.816497,0.852168,-0.617265,-0.586970,-0.935675,...,-0.551503,-0.406604,0.808493,-0.082052,1.007707,0.239081,-0.866997,0.302027,-0.963726,-1.017122
2,0.620766,0.816497,0.103057,-0.712873,0.714435,0.816497,0.852168,0.366289,2.250053,0.344258,...,-0.551503,-0.406604,-0.967742,-0.082052,-2.313146,-0.999015,-0.866997,-0.792820,-0.963726,-1.017122
3,-1.015799,-1.224745,-1.764843,-0.712873,-1.399708,0.816497,-1.173477,0.366289,-0.586970,0.344258,...,-0.551503,2.049964,0.808493,-0.082052,-0.652719,-0.999015,-0.095742,-0.792820,-0.963726,-1.017122
4,0.620766,0.816497,-1.764843,-0.712873,0.714435,0.816497,-1.173477,1.349843,-0.586970,2.904122,...,1.080495,-0.406604,-0.967742,-0.082052,-0.652719,-0.999015,-0.866997,-0.792820,-0.963726,-1.017122


Additionally, it is important to note that some of the numerical encodings for categories are nominal, not ordinal. In other words, some features do ***not** imply order*, meaning that other forms of encoding may be necessary at some point within the analysis. 

Addiitonally, just as one last confirmation that the data was correcltly loaded, we can observe the head of the dataframe:

In [8]:
hesp_df_raw.head()  # Display the first 5 rows of the dataframe

,Student Age,Sex,Graduated high-school type,Scholarship type,Additional work,Regular artistic or sports activity,Do you have a partner,Total salary if available,Transportation to the university,Accomodation type in Cyprus,...,Preparation to midterm exams 1,Preparation to midterm exams 2,Taking notes in classes,Listening in classes,Discussion improves my interest and success in the course,Flip-classroom,Cumulative grade point average in the last semester (/4.00),Expected Cumulative grade point average in the graduation (/4.00),Course ID,OUTPUT Grade
0,2,2,3,3,1,2,2,1,1,1,...,1,1,3,2,1,2,1,1,1,1
1,2,2,3,3,1,2,2,1,1,1,...,1,1,3,2,3,2,2,3,1,1
2,2,2,2,3,2,2,2,2,4,2,...,1,1,2,2,1,1,2,2,1,1
3,1,1,1,3,1,2,1,2,1,2,...,1,2,3,2,2,1,3,2,1,1
4,2,2,1,3,2,2,1,3,1,4,...,2,1,2,2,2,1,2,2,1,1


### Observing Feature Relationships

To help clearly define our research questions, we can firstly develop a correlation matrix to understand the relationships among features. This will give us insight on potential relationships among features and aid in modeling decisions.

In [27]:
import plotly.express as px  # Interactive visualizations

# Calculate the correlation matrix
corr_matrix = hesp_df_scaled.corr()

# Create the heatmap using Plotly
fig = px.imshow(
    corr_matrix,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Matrix of Student Performance Features'
)

# Update layout for better readability
fig.update_layout(width=1000, height=1000)

fig.show()

This very large correlation matrix provides immense insight on the different relationships among attributes. The only strong correlation (~65%) is between cumulative GPA from the last semester and expected cumulative GPA at graduation. This indicates more complex relationships that could use the assistance of ML strategies to help answer.

In [28]:
hesp_colnames = hesp_df_scaled.columns.tolist()  # Get column names of the dataset
print(hesp_colnames)  # Display column names

['Student Age', 'Sex', 'Graduated high-school type', 'Scholarship type', 'Additional work', 'Regular artistic or sports activity', 'Do you have a partner', 'Total salary if available', 'Transportation to the university', 'Accomodation type in Cyprus', "Mother's education", "Father's education", 'Number of sisters/brothers (if available)', 'Parental status', "Mother's occupation", "Father's occupation", 'Weekly study hours', 'Reading frequency (non-scientific books/journals)', 'Reading frequency (scientific books/journals)', 'Attendance to the seminars/conferences related to the department', 'Impact of your projects/activities on your success', 'Attendance to classes', 'Preparation to midterm exams 1', 'Preparation to midterm exams 2', 'Taking notes in classes', 'Listening in classes', 'Discussion improves my interest and success in the course', 'Flip-classroom', 'Cumulative grade point average in the last semester (/4.00)', 'Expected Cumulative grade point average in the graduation (/4

---

## Research and Methods

Now that the dataset has been explored and understood, we can begin articulating our curiosities and use relevant technical methods to answer them.

This analysis will focus on answering the following questions:

1) How can students be clustered? What groups can be derived from these clusters?
2) What does the "perfect student" look like, from this dataset? What are the optimal categories for acheiving a perfect output grade? Which indicators actually matter?
3) Which social indicators most correlate to attendance?

### 1. Investigating Student Clusters

For the first research question, it would be interesting to see what different groups of students there are, and how they could be clustered. From visualizations and descriptions of the clusters, we can make inferences on what these different student groups are.

Prior to running any clustering algorithms, we should remember that some features are nominal rather than ordinal. From manually observing the data and their labels, we can see that the following list of features are nominal:

```python
["Sex","Graduated high-school type","Additional work","Regular artistic or sports activity","Do you have a partner","Transportation to the university","Accomodation type in Cyprus","Parental status","Mother's occupation","Father's occupation","Attendance to the seminars/conferences related to the department","Preparation to midterm exams 1","Course ID"]
```

In order to use these features in our clustering algorithm, we can use One-HOT encoding. This will seperate the nominal features into their own usable columns, which we can pair with a clustering algorithm. To ensure correctness, we should do this on the initial raw dataframe, then scale it afterwards. 

In [32]:
# Explicitly defining nominal columns based on dataset documentation
hesp_nominal_columns = ["Sex","Graduated high-school type","Additional work","Regular artistic or sports activity","Do you have a partner",
                        "Transportation to the university","Accomodation type in Cyprus","Parental status","Mother's occupation","Father's occupation",
                        "Attendance to the seminars/conferences related to the department","Preparation to midterm exams 1","Course ID"]

# One-hot encode the nominal columns, creating a new dataframe
hesp_df_encoded = pd.get_dummies(hesp_df_raw, columns=hesp_nominal_columns)

# Scale numerical columns in the encoded dataframe
scaler_encoded = StandardScaler()
hesp_df_encoded_scaled = pd.DataFrame(
    scaler.fit_transform(hesp_df_encoded),
    columns=hesp_df_encoded.columns
)

# Display the new dataframe info to verify proper encoding
hesp_df_encoded_scaled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 65 columns):
 #   Column                                                              Non-Null Count  Dtype  
---  ------                                                              --------------  -----  
 0   Student Age                                                         145 non-null    float64
 1   Scholarship type                                                    145 non-null    float64
 2   Total salary if available                                           145 non-null    float64
 3   Mother's education                                                  145 non-null    float64
 4   Father's education                                                  145 non-null    float64
 5   Number of sisters/brothers (if available)                           145 non-null    float64
 6   Weekly study hours                                                  145 non-null    float64
 7   Reading frequency

Now that our dataframe is within the proper format, we can use a *DBSCAN* (Density-Based Spatial Clustering of Applications with Noise) to cluster students. Unlike k-Means or other algorithms, this one will automatically determine a number of clusters based on how many neighbors fall within a radius. This is acheived through the $\epsilon$ parameter (`eps`), which has been set to $9.0$ (after some testing) due to the large number of features.

In [43]:
from sklearn.decomposition import PCA

from sklearn.cluster import DBSCAN  # DBSCAN Clustering algorithm

# Fit the encoded data to the DBSCAN 'model'
dbscan = DBSCAN(eps=9.0, min_samples=4)  # Episilon scaled up due to noisy data and large number of features
dbscan.fit(hesp_df_encoded_scaled)

# Pull out the predicted cluster labels
cluster_labels = dbscan.labels_
unique_labels = set(cluster_labels)

# Print basic cluster information
print(f"Number of clusters found: {len(unique_labels) - (1 if -1 in cluster_labels else 0)}")
print(f"Number of noise points: {list(cluster_labels).count(-1)}")
print(f"Cluster labels: {unique_labels}")

# Add cluster labels to the encoded dataframe for visualization
hesp_df_encoded_scaled['Cluster'] = cluster_labels

# Create a 2D visualization using PCA for dimensionality reduction
pca = PCA(n_components=2)  # Reduce to 2 dimensions for plotting
pca_result = pca.fit_transform(hesp_df_encoded_scaled.drop('Cluster', axis=1))  # Get PCA result excluding cluster labels

# Create a dataframe for plotting
plot_df = pd.DataFrame({
    'PC1': pca_result[:, 0],
    'PC2': pca_result[:, 1],
    'Cluster': cluster_labels
})

# Create interactive scatter plot
fig = px.scatter(
    plot_df,
    x='PC1',
    y='PC2',
    color='Cluster',
    title='DBSCAN Clustering Results (PCA Projection)',
    labels={'PC1': 'First Principal Component', 'PC2': 'Second Principal Component'},
    color_continuous_scale='Viridis'
)

fig.update_traces(marker=dict(size=8))
fig.show()


Number of clusters found: 1
Number of noise points: 27
Cluster labels: {np.int64(0), np.int64(-1)}


Within this DBSCAN, only 1 cluster was found with a fairly high $\epsilon$ of $9.0$. This indicates that the relationships are fairly complex, and data points are very far apart.

Therefore, we can attempt this again with a k-Means clustering algorithm. We can use firstly use a sihoulette analysis to determine the optimal $k$.

In [47]:
from sklearn.metrics import silhouette_score  # Silhouette Score metric
from sklearn.cluster import KMeans  # KMeans Clustering algorithm

# Test different values of k
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(hesp_df_encoded_scaled.drop('Cluster', axis=1))
    silhouette_avg = silhouette_score(hesp_df_encoded_scaled.drop('Cluster', axis=1), cluster_labels)
    silhouette_scores.append(silhouette_avg)
    print(f"For k={k}, silhouette score = {silhouette_avg:.3f}")

# Plot silhouette scores
fig = px.line(
    x=list(k_range), 
    y=silhouette_scores, 
    markers=True,
    labels={'x': 'Number of Clusters (k)', 'y': 'Silhouette Score'},
    title='Silhouette Analysis for Optimal k'
)
fig.show()

# Find optimal k
optimal_k = list(k_range)[silhouette_scores.index(max(silhouette_scores))]
print(f"\nOptimal number of clusters: {optimal_k} (silhouette score = {max(silhouette_scores):.3f})")

For k=2, silhouette score = 0.110
For k=3, silhouette score = 0.072
For k=4, silhouette score = 0.051
For k=5, silhouette score = 0.036
For k=6, silhouette score = 0.035
For k=7, silhouette score = 0.038
For k=5, silhouette score = 0.036
For k=6, silhouette score = 0.035
For k=7, silhouette score = 0.038
For k=8, silhouette score = 0.034
For k=9, silhouette score = 0.045
For k=10, silhouette score = 0.050
For k=8, silhouette score = 0.034
For k=9, silhouette score = 0.045
For k=10, silhouette score = 0.050



Optimal number of clusters: 2 (silhouette score = 0.110)


These scores indicate that clusters for this dataset will be very broad, having very weak partitions. Regardless, $k=2$ had a clear best score of around $.11$, indicating the two clusters will be optimal for determining any sort of groupings among students. Using this, we can assign labels using the k-Means algorithm and visualize the results to attempt to derive any meaning.

In [49]:
#Instantiate and fit the optimal KMeans model
hesp_kmeans = KMeans(n_clusters=optimal_k, random_state=42)
hesp_kmeans.fit(hesp_df_encoded_scaled.drop('Cluster', axis=1))

# Pull out the predicted cluster labels
cluster_labels = hesp_kmeans.labels_
unique_labels = set(cluster_labels)

# Print basic cluster information
print(f"Number of clusters found: {len(unique_labels) - (1 if -1 in cluster_labels else 0)}")
print(f"Number of noise points: {list(cluster_labels).count(-1)}")
print(f"Cluster labels: {unique_labels}")

# Add cluster labels to the encoded dataframe for visualization
hesp_df_encoded_scaled['Cluster'] = cluster_labels

# Create a 2D visualization using PCA for dimensionality reduction
pca = PCA(n_components=2)  # Reduce to 2 dimensions for plotting
pca_result = pca.fit_transform(hesp_df_encoded_scaled.drop('Cluster', axis=1))  # Get PCA result excluding cluster labels

# Create a dataframe for plotting
plot_df = pd.DataFrame({
    'PC1': pca_result[:, 0],
    'PC2': pca_result[:, 1],
    'Cluster': cluster_labels
})

# Create interactive scatter plot
fig = px.scatter(
    plot_df,
    x='PC1',
    y='PC2',
    color='Cluster',
    title='DBSCAN Clustering Results (PCA Projection)',
    labels={'PC1': 'First Principal Component', 'PC2': 'Second Principal Component'},
    color_continuous_scale='Viridis'
)

fig.update_traces(marker=dict(size=8))
fig.show()


Number of clusters found: 2
Number of noise points: 0
Cluster labels: {np.int32(0), np.int32(1)}


On a 2D plane of the 2 most significant principle components, these clusters appear much more seperated. To better understand the actual differences among these clusters, we can compare the group averages across all features. 

In [52]:
cluster_profiles = hesp_df_encoded_scaled.groupby('Cluster').mean()  # Find the mean (by cluster) for each feature
cluster_profiles.T  # Transpose for better readability

Cluster,0,1
Student Age,0.094219,-0.361173
Scholarship type,-0.030621,0.117381
Total salary if available,-0.044238,0.169578
Mother's education,-0.103572,0.397024
Father's education,-0.030153,0.115588
...,...,...
Course ID_5,0.058753,-0.225221
Course ID_6,0.024953,-0.095653
Course ID_7,0.060060,-0.230230
Course ID_8,-0.179699,0.688846


To see exactly what features differed substantially, we can run a *Mann-Whitney U* test to find statistically significant differences between clusters.

In [64]:
from scipy.stats import mannwhitneyu  # For statistical significance testing

# Store Mann-Whitney U test results for each feature
cluster_significance_results = []

# Iterate through each column in the encoded and scaled dataframe
for col in hesp_df_encoded_scaled.columns:
    
    if col != 'Cluster':  # Skip the cluster label column itself
        # Extract data for the two clusters
        group0 = hesp_df_encoded_scaled.loc[hesp_df_encoded_scaled["Cluster"] == 0, col]
        group1 = hesp_df_encoded_scaled.loc[hesp_df_encoded_scaled["Cluster"] == 1, col]
        # Perform Mann-Whitney U test
        stat, p_value = mannwhitneyu(group0, group1, alternative='two-sided')
        # Store the p-value for this feature and cluster combination
        cluster_significance_results.append((col, p_value))
            
# Convert results to a DataFrame for easier interpretation
cluster_significance_results_df = pd.DataFrame(cluster_significance_results, columns=["feature", "p_value"]).sort_values("p_value")

# Print the counts of each cluster (for interpretation)
print(hesp_df_encoded_scaled['Cluster'].value_counts())

# Show all features with p-value less than 0.05, sorted by significance
sig_features = (cluster_significance_results_df[cluster_significance_results_df["p_value"] < 0.05].sort_values("p_value"))
sig_features

Cluster
0    115
1     30
Name: count, dtype: int64


,feature,p_value
20,Sex_2,1.374330e-12
19,Sex_1,1.374330e-12
51,Attendance to the seminars/conferences related...,1.316505e-11
52,Attendance to the seminars/conferences related...,1.316505e-11
64,Course ID_9,6.346885e-10
42,Mother's occupation_2,2.999800e-08
36,Accomodation type in Cyprus_3,4.712555e-08
31,Transportation to the university_2,1.086983e-07
30,Transportation to the university_1,7.383030e-06
56,Course ID_1,1.250962e-05


These results indicate that the clusters are strongly seperated by numerous factors, including gender, academic participation behaviors, and some socioeconomic/personal contexts. By looking at these cluster means, we can then determine which cluster correlates to the different groups of students.

In [66]:
significant_means = cluster_profiles[sig_features['feature']]  # Filter the cluster profiles to show only the significant features
significant_means.T  # Transpose for better readability

Cluster,0,1
Sex_2,0.301749,-1.156703
Sex_1,-0.301749,1.156703
Attendance to the seminars/conferences related to the department_1,0.288161,-1.104617
Attendance to the seminars/conferences related to the department_2,-0.288161,1.104617
Course ID_9,-0.263275,1.009222
Mother's occupation_2,0.235992,-0.904636
Accomodation type in Cyprus_3,-0.232625,0.891729
Transportation to the university_2,-0.226233,0.867227
Transportation to the university_1,0.190909,-0.731818
Course ID_1,0.186057,-0.713217


By interpreting these results as represented by the original labels, we can derive the following table describing the cluster differences:

<!-- Table describing cluster differences -->
| Dimension                             | **Cluster 0 Profile**                                                | **Cluster 1 Profile**                                                 |
| ------------------------------------- | -------------------------------------------------------------------- | --------------------------------------------------------------------- |
| **Gender distribution**               | Mostly female                                           | Mostly male                                              |
| **Academic engagement**               | Higher attendance at seminars/conferences                            | Lower attendance at seminars/conferences                              |
| **Course grouping**                   | More associated with Course ID 1                                     | More associated with Course IDs 8 and 9                               |
| **Academic performance**              | Higher previous GPA, expected GPA, and final grades                  | Lower previous GPA, expected GPA, and final grades                    |
| **Study behavior**                    | More individual exam preparation & more extracurricular participation | Different preparation tendencies & lower extracurricular participation |
| **Perceived impact of learning**      | Higher belief that projects/activities support success               | Lower perceived benefit from academic activities                      |

Therefore, we can conclude that this dataset **can be cleanly seperated into two distinct groups**. The first group reflects academically engaged and higher-performing students, being predominantly female, attending more seminars, having a higher GPA, and stronger study habits. The second group, on the other hand, reflects less engagement and lower-performing students, being predominantly male, and having lower attendance.

### 2. Modeling the Perfect Student

TODO: For this one, will train a model and use the best features attribute or whatever. This way, we could run PCA before hand easily to get the best accuracy and narrow down, run a grid search to optimize, then pull out 4-5 best feature estimators (most significant).

NOTE: Could do the same thing for question 3 but with an ANN. Just make this brief though and dependent on time.